# Attrition risk analysis — starter notebook

This notebook builds an interpretable attrition risk score from HR-available signals (no black-box model) and surfaces which teams and risk factors need attention first. It is designed to be explainable to a manager, not just accurate on paper.

**What this notebook does:**
1. Loads employee tenure, engagement, and career-velocity signals
2. Computes an interpretable risk score per employee (not a black-box probability)
3. Aggregates risk by team, manager, and tenure band
4. Identifies which risk factors are driving the most exposure org-wide
5. Flags equity checks that must run before this is used to target retention spend

**Data requirements:**
- Tenure and role history (from HRIS)
- Engagement survey scores, most recent cycle (from engagement platform)
- Manager change history (from HRIS)
- Time since last promotion or level change (from HRIS)
- Compensation-to-market-ratio, if available (from comp system) — optional but improves accuracy

**Before using in production:** Complete the [Risk Assessment Template](../03-governance/risk-assessment-template.md). Attrition scoring is a sensitive use case — scores should go to HRBPs for retention conversation planning, never directly to managers as a ranked list, and never used as a factor in performance or compensation decisions.

---

In [ ]:
# Requirements
# pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Define risk factors and weights

Replace with weights calibrated to your organization's actual attrition drivers (ideally derived from a regression or survival analysis against historical departures — these starting weights are illustrative, not empirical).

**Risk factors used here:**
- Engagement score (lower = higher risk)
- Months since last promotion or level change (longer = higher risk, with diminishing effect)
- Manager changes in the last 18 months (more = higher risk)
- Compensation-to-market ratio (below 1.0 = higher risk)
- Tenure in current role relative to typical time-in-role for that role

In [ ]:
# ── REPLACE WITH WEIGHTS CALIBRATED TO YOUR ATTRITION DATA ───────────────────
# These are illustrative starting weights, not derived from real attrition data.
# Calibrate against your own historical departures before using for real prioritization.

RISK_WEIGHTS = {
    'low_engagement': 0.30,
    'promotion_stall': 0.20,
    'manager_churn': 0.20,
    'comp_below_market': 0.20,
    'role_overtenure': 0.10,
}
# ─────────────────────────────────────────────────────────────────────────────

assert abs(sum(RISK_WEIGHTS.values()) - 1.0) < 1e-9, 'weights must sum to 1.0'
print('Risk factors and weights:')
for factor, weight in RISK_WEIGHTS.items():
    print(f'  {factor}: {weight}')

## 2. Load employee signals

Replace the synthetic data generator with your actual HRIS, engagement platform, and comp system exports, joined on employee_id.

In [ ]:
# ── REPLACE WITH YOUR HRIS / ENGAGEMENT / COMP EXPORTS, JOINED ON employee_id ─

def generate_synthetic_signals(n=200):
    """Synthetic employee risk signals for development only."""
    teams = ['Engineering', 'People', 'Sales', 'Customer Success', 'Finance']
    team_weights = [0.35, 0.15, 0.25, 0.15, 0.10]
    managers_per_team = 6

    records = []
    for i in range(n):
        team = np.random.choice(teams, p=team_weights)
        manager_id = f'{team[:3].upper()}-MGR{np.random.randint(1, managers_per_team + 1)}'
        tenure_months = int(np.random.gamma(3, 14))
        engagement_score = float(np.clip(np.random.normal(3.6, 0.7), 1.0, 5.0))
        months_since_promo = int(np.random.gamma(2, 10))
        manager_changes_18mo = np.random.choice([0, 1, 2, 3], p=[0.55, 0.25, 0.15, 0.05])
        comp_ratio = float(np.clip(np.random.normal(1.02, 0.12), 0.65, 1.35))
        typical_time_in_role = 24
        records.append({
            'employee_id': f'EMP{i+1:04d}',
            'team': team,
            'manager_id': manager_id,
            'tenure_months': tenure_months,
            'engagement_score': round(engagement_score, 2),
            'months_since_promo': months_since_promo,
            'manager_changes_18mo': manager_changes_18mo,
            'comp_ratio': round(comp_ratio, 2),
            'time_in_role_months': min(tenure_months, int(np.random.gamma(2.5, 10))),
            'typical_time_in_role': typical_time_in_role,
        })
    return pd.DataFrame(records)

df = generate_synthetic_signals()
# ─────────────────────────────────────────────────────────────────────────────

print(f'Employees: {len(df):,}')
print(f'Team distribution:')
print(df['team'].value_counts())

## 3. Compute interpretable risk score

Each factor is normalized to a 0-1 sub-score, then combined using the weights above. The output is a transparent, explainable score — every component can be shown to an HRBP, unlike a black-box model output.

In [ ]:
def score_engagement(val):
    """Lower engagement = higher risk. Scale 1-5 inverted to 0-1."""
    return float(np.clip((5.0 - val) / 4.0, 0, 1))

def score_promotion_stall(months):
    """Risk climbs with months since promotion, plateaus after 36 months."""
    return float(np.clip(months / 36.0, 0, 1))

def score_manager_churn(changes):
    """Each manager change in 18mo adds risk, capped at 3."""
    return float(np.clip(changes / 3.0, 0, 1))

def score_comp(ratio):
    """Risk only accrues below market (ratio < 1.0). At or above market = 0 risk from this factor."""
    return float(np.clip((1.0 - ratio) / 0.35, 0, 1)) if ratio < 1.0 else 0.0

def score_role_overtenure(time_in_role, typical):
    """Risk climbs once time in role exceeds typical time in role for that role type."""
    return float(np.clip((time_in_role - typical) / typical, 0, 1)) if time_in_role > typical else 0.0

df['sub_engagement'] = df['engagement_score'].apply(score_engagement)
df['sub_promo_stall'] = df['months_since_promo'].apply(score_promotion_stall)
df['sub_manager_churn'] = df['manager_changes_18mo'].apply(score_manager_churn)
df['sub_comp'] = df['comp_ratio'].apply(score_comp)
df['sub_overtenure'] = df.apply(lambda r: score_role_overtenure(r['time_in_role_months'], r['typical_time_in_role']), axis=1)

df['risk_score'] = (
    RISK_WEIGHTS['low_engagement'] * df['sub_engagement'] +
    RISK_WEIGHTS['promotion_stall'] * df['sub_promo_stall'] +
    RISK_WEIGHTS['manager_churn'] * df['sub_manager_churn'] +
    RISK_WEIGHTS['comp_below_market'] * df['sub_comp'] +
    RISK_WEIGHTS['role_overtenure'] * df['sub_overtenure']
).round(3)

df['risk_tier'] = pd.cut(
    df['risk_score'],
    bins=[0, 0.25, 0.5, 1.01],
    labels=['Low', 'Moderate', 'High']
)

print('Risk score distribution:')
print(df['risk_score'].describe().round(3))
print()
print(df['risk_tier'].value_counts())

## 4. Which factor is driving risk, org-wide

A risk score is only useful for retention planning if you know *why* it's high. This breaks down average contribution by factor so the program can target root causes, not just flag names.

In [ ]:
weighted_contributions = pd.DataFrame({
    'factor': list(RISK_WEIGHTS.keys()),
    'avg_weighted_contribution': [
        (RISK_WEIGHTS['low_engagement'] * df['sub_engagement']).mean(),
        (RISK_WEIGHTS['promotion_stall'] * df['sub_promo_stall']).mean(),
        (RISK_WEIGHTS['manager_churn'] * df['sub_manager_churn']).mean(),
        (RISK_WEIGHTS['comp_below_market'] * df['sub_comp']).mean(),
        (RISK_WEIGHTS['role_overtenure'] * df['sub_overtenure']).mean(),
    ]
}).sort_values('avg_weighted_contribution', ascending=False)

print('Org-wide risk drivers, by average weighted contribution:')
print(weighted_contributions.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(weighted_contributions['factor'], weighted_contributions['avg_weighted_contribution'],
               color='#185FA5', alpha=0.85)
ax.set_xlabel('Average contribution to risk score')
ax.set_title('What is driving attrition risk, org-wide', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('attrition_risk_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Risk concentration by team and manager

High-risk employees clustered under one manager is a different problem (and a different intervention) than high risk spread evenly across a team.

In [ ]:
team_risk = df.groupby('team')['risk_score'].agg(['mean', 'count']).round(3)
team_risk.columns = ['avg_risk_score', 'headcount']
team_risk = team_risk.sort_values('avg_risk_score', ascending=False)
print('Average risk score by team:')
print(team_risk.to_string())
print()

manager_risk = df.groupby('manager_id').agg(
    avg_risk_score=('risk_score', 'mean'),
    high_risk_reports=('risk_tier', lambda s: (s == 'High').sum()),
    headcount=('risk_score', 'count')
).round(3)
manager_risk = manager_risk[manager_risk['headcount'] >= 3].sort_values('avg_risk_score', ascending=False)
print('Managers with 3+ reports, ranked by average team risk (top 10):')
print(manager_risk.head(10).to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x=team_risk.index, y=team_risk['avg_risk_score'], ax=ax, color='#185FA5', alpha=0.85)
ax.set_ylabel('Average risk score')
ax.set_xlabel('')
ax.set_title('Average attrition risk by team', fontsize=13)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('attrition_risk_by_team.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Retention outreach prioritization

Exports a prioritized list for HRBP outreach planning. This output should go to HRBPs for retention conversations, not to line managers as a ranked list — a manager receiving a ranked flight-risk list on their own team creates its own morale and fairness problems.

In [ ]:
priority_output = df[df['risk_tier'] == 'High'][[
    'employee_id', 'team', 'manager_id', 'risk_score',
    'sub_engagement', 'sub_promo_stall', 'sub_manager_churn', 'sub_comp', 'sub_overtenure'
]].sort_values('risk_score', ascending=False)

print(f'High-risk employees flagged for HRBP outreach: {len(priority_output)}')
print(priority_output.head(15).to_string(index=False))

priority_output.to_csv('attrition_retention_priorities.csv', index=False)
print('\nExported to: attrition_retention_priorities.csv')
print('Distribution: HRBPs only. Do not share ranked individual scores with line managers.')

## Next steps

Before using this analysis to drive retention spend or outreach:

1. **Run the equity check first, not last.** Break down average risk score by demographic group (where lawful to analyze) before acting on this data. If one group scores systematically higher, investigate whether the underlying signals (promotion velocity, comp ratio) reflect a fairness issue in those systems rather than a genuine flight-risk difference.

2. **Calibrate the weights against real departures.** The weights in this notebook are illustrative. Before using scores to prioritize spend, validate them against 12-24 months of actual attrition data for your organization — a simple logistic regression against historical departures will tell you whether these five factors, and these weights, actually predict who left.

3. **Never treat the score as a diagnosis of intent.** A high score means "this person has more of the risk factors we know correlate with departure," not "this person is planning to leave." Treat it as a prompt for a genuine retention conversation, not a prediction to act on unilaterally.

4. **Restrict distribution.** Scores go to HRBPs for outreach planning. They should never be visible to the employee's manager as a ranked list, never factor into performance or compensation decisions, and never be used to justify a decision that should stand on its own evidence.

5. **Re-score on a cadence, not continuously.** Monthly or quarterly re-scoring is enough signal for outreach planning. Continuous real-time scoring invites overuse and treats a nuanced human situation like a live dashboard metric.

6. **Route through governance before deployment.** Complete the [risk assessment template](../03-governance/risk-assessment-template.md) and, for EU-based employees, the [EU AI Act intake template](../03-governance/eu-ai-act-intake-template.md) — attrition scoring that influences HR decisions about individuals is squarely in scope.